In [1]:
from symmer.operators.base import PauliwordOp, QuantumState
import h5py
import numpy as np 
from pyscf import scf
import math

In [2]:
chkfile = "O2.h5"

# load chemistry data
mol_load = scf.chkfile.load_mol(chkfile)
scf_load = scf.ROHF(mol_load) 
scf_load.__dict__.update(scf.chkfile.load(chkfile, 'scf')) 

# print chemistry details (note other info can be accessed in these objects see PySCF for further details.)
print(f"n spin orbitals: {2*mol_load.nao}")
print(f"basis: {mol_load.basis}")
xyz_str = mol_load.tostring(format="xyz")
print(f"geometry:\n\n{xyz_str}")
print(f"unit: {mol_load.unit}")
print(f"initaliziation: {scf_load.init_guess}")
E_rohf = scf_load.e_tot
print(f"e_rohf: {E_rohf}")

n spin orbitals: 20
basis: sto-3g
geometry:

2
XYZ from PySCF
O           0.00000000        0.00000000        0.00000000
O           0.00000000        0.00000000        2.55000000
unit: angstrom
initaliziation: minao
e_rohf: -147.46911791564932


In [3]:
serach_space = math.comb(mol_load.nao, mol_load.nelec[0]) * math.comb(mol_load.nao, mol_load.nelec[1]) 
serach_space

1200

In [4]:
with h5py.File("O2_data.h5", "r") as infile:
    # print(infile.keys())
    H = PauliwordOp(infile["H_symp"][()], infile["H_coeff"][()])

    hf_psi = QuantumState(infile["hf_state_mat"][()], infile["hf_coeff"][()])
    
    fci_psi = QuantumState(infile["fci_state_mat"][()], infile["fci_state_coeff"][()])
    e_fci = infile["e_fci"][()][0]

In [5]:
E_hf = H.expval(hf_psi)
E_hf

np.complex128(-147.46911791564935+0j)

In [6]:
E_fci = H.expval(fci_psi)
E_fci

np.complex128(-147.60887634990078+0j)

In [7]:
print(f"energy ROHF: {E_hf.real:0.9f}")
print(f"energy fci : {E_fci.real:0.9f}")

energy ROHF: -147.469117916
energy fci : -147.608876350


In [8]:
fci_rohf      = (fci_psi.dagger * hf_psi)
assert fci_rohf.imag == 0
fci_rohf = fci_rohf.real

fci_rohf_ovlp = np.abs(fci_rohf)**2

In [9]:
fci_rohf

np.float64(2.1707849162668412e-07)

In [10]:
print(f" < FCI | ROHF >    = {fci_rohf:0.16f}")
print(f"|< FCI | ROHF >|^2 = {fci_rohf_ovlp:0.16f}")

 < FCI | ROHF >    = 0.0000002170784916
|< FCI | ROHF >|^2 = 0.0000000000000471


In [12]:
fci_sorted = fci_psi.sort() # <-- sort state by coefficent magnitude (largest first)

fci_sorted[:20] #<-- print top 20 terms for space

-0.354+0.000j |11111111110111101010> +
-0.354+0.000j |11111111111011101001> +
-0.354+0.000j |11111111111011100110> +
-0.354+0.000j |11111111111011011010> +
-0.339+0.000j |11111111101001101111> +
-0.339+0.000j |11111111100110101111> +
-0.338+0.000j |11111111101010011111> +
-0.338+0.000j |11111111011010101111> +
-0.094+0.000j |11111111101001111011> +
-0.094+0.000j |11111111100110111011> +
-0.094+0.000j |11111111101010110111> +
-0.094+0.000j |11111111011010111011> +
 0.036+0.000j |11111111110110101110> +
 0.036+0.000j |11111111111001101110> +
 0.036+0.000j |11111111111010101101> +
 0.036+0.000j |11111111111010011110> +
 0.010+0.000j |11111111111001111010> +
 0.010+0.000j |11111111110110111010> +
 0.010+0.000j |11111111111010111001> +
 0.010+0.000j |11111111111010110110>

In [13]:
# look over dominate ampliitude terms and measure energy

for i in range(4):
    psi = QuantumState(fci_sorted[i].state_matrix, [1])
    print("psi:", psi)
    e_psi = H.expval(psi).real
    print(e_psi)
    psi_fci = psi.dagger * fci_psi
    assert psi_fci.imag ==0
    psi_fci = psi_fci.real
    print(f" < psi | FCI > = {psi_fci:0.16f}")
    print(f"|< psi | FCI >|^2 = {np.abs(psi_fci)**2:0.16f}")
    print()

psi:  1.000+0.000j |11111111110111101010>
-147.56068060745406
 < psi | FCI > = -0.3542634752259560
|< psi | FCI >|^2 = 0.1255026098791716

psi:  1.000+0.000j |11111111111011101001>
-147.56023564699106
 < psi | FCI > = -0.3542435745167302
|< psi | FCI >|^2 = 0.1254885100863902

psi:  1.000+0.000j |11111111111011100110>
-147.5603762390445
 < psi | FCI > = -0.3535639962352000
|< psi | FCI >|^2 = 0.1250074994338045

psi:  1.000+0.000j |11111111111011011010>
-147.56033160363634
 < psi | FCI > = -0.3535549815180370
|< psi | FCI >|^2 = 0.1250011249562195



In [14]:
int("11111111110111101010",2)

1048042

In [15]:
Hamiltonian_dictionary = H.to_dictionary

terms = list(Hamiltonian_dictionary.items())[:10] #<-- for space only print first 10 terms (use dictionary for access to all)
for P, coeff in terms:
    print(coeff, P)

(-92.17826271664114+0j) IIIIIIIIIIIIIIIIIIII
(12.731304493534395+0j) ZIIIIIIIIIIIIIIIIIII
(12.731304493534395+0j) IZIIIIIIIIIIIIIIIIII
(-4.894347564516634e-06+0j) XZXIIIIIIIIIIIIIIIII
(-4.894347564516634e-06+0j) YZYIIIIIIIIIIIIIIIII
(-4.894347564516634e-06+0j) IXZXIIIIIIIIIIIIIIII
(-4.894347564516634e-06+0j) IYZYIIIIIIIIIIIIIIII
(0.1323246565260403+0j) XZZZXIIIIIIIIIIIIIII
(0.1323246565260403+0j) YZZZYIIIIIIIIIIIIIII
(0.1323246565260403+0j) IXZZZXIIIIIIIIIIIIII
